# Kodra AI Agent: Cloud/Colab GPU Training Preparation Notebook

**Product:** Kodra AI Agent
**Model:** Kodra GPT (`KodraGPT`)
**Core:** Kodra Core
**Tagline:** CODE - THINK - CREATE

This notebook prepares and validates a GPU training run for **Kodra GPT** on Google Colab (or any
Jupyter environment with a CUDA GPU). Every cell is labeled `CELL NN` and runs in a fixed,
sequential order: clone -> install -> verify hardware -> build/validate a clean dataset -> verify
no metadata contamination -> tokenizer -> model selection -> dataloaders -> causal-shift proof ->
a 20-step GPU smoke test -> checkpoint save/reload/resume -> generation/syntax evaluation ->
optional Drive backup. Full multi-epoch training lives behind an explicit **RUN MANUALLY ONLY**
gate in the final cell and never runs as part of "Run All".

### Why earlier runs produced malformed Python and a literal `target:` in generated text

A prior diagnostic pass flagged the substring `target:` inside the training corpus as "leaked
metadata" (as if it were a serialized `{"target": ...}` record field). That diagnosis was a
**false positive**: the only place `target:` appears in the corpus is `data/code/algorithms.py`'s
binary-search implementation - `target` is an ordinary Python parameter name, and the `:` is the
block colon of `if arr[mid] == target:` / `elif arr[mid] < target:`. That is completely valid,
intentional Python, not contamination. `CELL 07` below proves this with a validator that
distinguishes real (quoted-key) metadata leakage from legitimate code, and both this notebook and
`datasets/corpus_pipeline.py` now reject the former while never flagging the latter.

The real reason generated Python was malformed and 0% syntax-parse rate was observed is plain
**severe undertraining**: a ~700-1200 character corpus, a BPE vocabulary that only reached ~415
tokens against a target of 8000, and 20-50 optimizer steps are nowhere near enough for a model to
learn Python syntax. `def quicksort(arr):target:` is the model regurgitating a memorized fragment
of the binary-search function it saw a handful of times, glued onto a different prompt - a
symptom of too little data and too few steps, not a data-pipeline bug. No further serious training
should happen until a larger, validated corpus (`CELL 06`) is used, per Phase F/H.

No credentials are embedded in this notebook. If you want to persist checkpoints to Google Drive,
mount it yourself in Colab and pass that path as `CHECKPOINT_DIR` below.

In [3]:
# CELL 01 - Clone the repository (HTTPS, works unauthenticated on Colab) and enter kodra-core
!rm -rf /content/Kodra-ai
!git clone https://github.com/ChaceEthan/Kodra-ai.git /content/Kodra-ai
%cd /content/Kodra-ai/kodra-core

Cloning into '/content/Kodra-ai'...
remote: Enumerating objects: 246, done.
remote: Counting objects: 100% (246/246), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 246 (delta 97), reused 205 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (246/246), 206.29 KiB | 1.11 MiB/s, done.
Resolving deltas: 100% (97/97), done.
/content/Kodra-ai/kodra-core


In [ ]:
# List files in the current directory to confirm repository structure
!ls -F

agent/	   evaluation/	notebooks/	  scripts/    training/
configs/   inference/	README.md	  server/     vscode-extension/
data/	   LICENSE	requirements.txt  tests/
datasets/  model/	ROADMAP.md	  tokenizer/


In [5]:
# CELL 02 - Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# CELL 03 - Verify CUDA is available
import torch

CUDA_AVAILABLE = torch.cuda.is_available()
print('CUDA available:', CUDA_AVAILABLE)
if not CUDA_AVAILABLE:
    print('No GPU detected - training will fall back to CPU (slow for anything above kodra-tiny).')

CUDA available: True


In [ ]:
# CELL 04 - Print GPU name and VRAM
if CUDA_AVAILABLE:
    print('Device name:', torch.cuda.get_device_name(0))
    print('Device count:', torch.cuda.device_count())
    print('Total memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print('Skipping GPU info - no CUDA device detected.')

Device name: Tesla T4
Device count: 1
Total memory (GB): 15.64


In [ ]:
# CELL 05 - (Optional) Mount Google Drive for persistent checkpoint storage.
# Uncomment if you want checkpoints to survive a Colab session restart.
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/kodra_checkpoints'
# CHECKPOINT_DIR = 'checkpoints'
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)


Mounted at /content/drive
CHECKPOINT_DIR: /content/drive/MyDrive/kodra_checkpoints


In [1]:
import os

DATASET_DIR = "/content/drive/MyDrive/Kodra-training-data"

print("DATASET:", DATASET_DIR)
print("Exists:", os.path.isdir(DATASET_DIR))

files = []
for root, dirs, filenames in os.walk(DATASET_DIR):
    for f in filenames:
        files.append(os.path.relpath(os.path.join(root, f), DATASET_DIR))

print("Total files:", len(files))
print("\nFirst 30 files:")
for f in files[:30]:
    print(f)


DATASET: /content/drive/MyDrive/Kodra-training-data
Exists: False
Total files: 0

First 30 files:


In [2]:
MANIFEST_PATH = os.path.join(DATASET_DIR, '.manifest/manifest.json')
print(f'\nChecking for manifest file: {MANIFEST_PATH}')
print(f'Manifest exists: {os.path.exists(MANIFEST_PATH)}')



Checking for manifest file: /content/drive/MyDrive/Kodra-training-data/.manifest/manifest.json
Manifest exists: False


In [9]:
# CELL 06 - Fresh-training configuration + build/validate the approved dataset.
#
# This block is the single source of truth for the rest of the notebook. Only
# APPROVED_DATASET_DIR is meant to change between runs - point it at a directory you have
# explicitly approved for training. This notebook never downloads datasets or clones other
# repositories; no external/private project path is hardcoded here. The actual safety rules
# (fresh_training forces fresh_tokenizer/blocks the legacy checkpoint dir, ...) live in
# training/fresh_mode.py, which is unit-tested directly - this cell only sets the raw flags.
FRESH_TRAINING = True                # master switch: start a fresh model/tokenizer/checkpoint lineage
APPROVED_DATASET_DIR = "/content/drive/MyDrive/Kodra-training-data" # <-- point this at your approved corpus; 'data/code' is only the bundled 733-char sample
FRESH_TOKENIZER = True               # train a brand-new tokenizer from APPROVED_DATASET_DIR instead of loading an old one
USE_OLD_CHECKPOINT = False           # if True, would resume the legacy diagnostic checkpoint

from training.fresh_mode import resolve_fresh_training_config

fresh_cfg = resolve_fresh_training_config(
    fresh_training=FRESH_TRAINING,
    approved_dataset_dir=APPROVED_DATASET_DIR,
    fresh_tokenizer=FRESH_TOKENIZER,
    use_old_checkpoint=USE_OLD_CHECKPOINT,
)
if fresh_cfg.fresh_training and (not FRESH_TOKENIZER or USE_OLD_CHECKPOINT):
    print('FRESH_TRAINING=True overrides FRESH_TOKENIZER/USE_OLD_CHECKPOINT to their safe values.')

# Fresh runs get their own checkpoint lineage and tokenizer file, completely separate from the
# legacy/diagnostic artifacts under checkpoints/ and tokenizer/vocab_bpe.json - a fresh run can
# never overwrite or silently resume from those.
CHECKPOINT_DIR = fresh_cfg.checkpoint_dir
LEGACY_CHECKPOINT_DIR = fresh_cfg.checkpoint_dir  # diagnostic-only; never read/written under FRESH_TRAINING
BPE_VOCAB_PATH = fresh_cfg.tokenizer_path

print(f'FRESH_TRAINING={fresh_cfg.fresh_training} | APPROVED_DATASET_DIR={fresh_cfg.approved_dataset_dir!r} | '
      f'FRESH_TOKENIZER={fresh_cfg.fresh_tokenizer} | USE_OLD_CHECKPOINT={fresh_cfg.use_old_checkpoint}')
print(f'CHECKPOINT_DIR={CHECKPOINT_DIR}')
print(f'BPE_VOCAB_PATH={BPE_VOCAB_PATH}')

# `build_manifest` only ever reads a local, explicitly-approved directory (never the internet or
# an arbitrary repo) and rejects bad records before they can reach training: serialized metadata
# (quoted "target": / "prompt": keys), U+FFFD corruption, and minified/generated code - see
# DEFAULT_QUALITY_FILTERS in datasets/corpus_pipeline.py. To scale beyond the bundled sample
# corpus, run the same pipeline from the command line against your own approved directory:
#   python scripts/prepare_training_corpus.py --source <approved-directory> --output data/manifest.json
# and re-point APPROVED_DATASET_DIR above at that directory - no other code changes needed.
from datasets.corpus_pipeline import build_manifest, write_manifest, build_training_text

MANIFEST_PATH = 'data/manifest.json'
manifest = build_manifest(fresh_cfg.approved_dataset_dir, seed=42, val_ratio=0.1, test_ratio=0.0,
                           license='project-sample', source='kodra-approved-corpus')
write_manifest(manifest, MANIFEST_PATH)

# TRAIN_TEXT_CANDIDATE / VAL_TEXT_CANDIDATE are assembled ONLY from the manifest's clean,
# per-file deterministic train/val split (build_training_text re-reads each file fresh from
# disk) - manifest bookkeeping fields (license, source, sha256, ...) never enter the text the
# model actually trains on.
TRAIN_TEXT_CANDIDATE = build_training_text(manifest, split='train')
VAL_TEXT_CANDIDATE = build_training_text(manifest, split='val')

print(f'Discovered {manifest.num_files} files, {manifest.total_chars} chars, '
      f'{manifest.total_token_estimate} approx tokens')
print(f'Languages: {manifest.language_counts}')
print(f'Train/val files: {manifest.split_counts["train"]}/{manifest.split_counts["val"]}')
print(f'Rejected - duplicates: {manifest.num_duplicates_removed}, '
      f'encoding: {manifest.num_encoding_rejected}, '
      f'secrets: {manifest.num_secrets_redacted}, '
      f'quality/contamination: {manifest.num_filtered_out} {manifest.filtered_reasons}')

if manifest.total_token_estimate < 1_000_000:
    print(f'\nNOTE: ~{manifest.total_token_estimate:,} tokens is below the 1,000,000-token minimum '
          f'for a serious training run (see training/readiness.py and CELL 21). Smoke-testing the '
          f'pipeline on this corpus is fine; do not flip RUN_FULL_TRAINING on it.')


FRESH_TRAINING=True | APPROVED_DATASET_DIR='/content/drive/MyDrive/Kodra-training-data' | FRESH_TOKENIZER=True | USE_OLD_CHECKPOINT=False
CHECKPOINT_DIR=checkpoints/fresh_training
BPE_VOCAB_PATH=tokenizer/vocab_bpe_fresh.json
Discovered 0 files, 0 chars, 0 approx tokens
Languages: {}
Train/val files: 0/0
Rejected - duplicates: 0, encoding: 0, secrets: 0, quality/contamination: 0 {}

NOTE: ~0 tokens is below the 1,000,000-token minimum for a serious training run (see training/readiness.py and CELL 21). Smoke-testing the pipeline on this corpus is fine; do not flip RUN_FULL_TRAINING on it.


In [ ]:
# CELL 07 - Verify no metadata contamination.
# This replaces the old naive `'target:' in TRAIN_TEXT` substring check, which produced a false
# positive on legitimate code (see the notebook intro). The real check requires a QUOTED key -
# "target": / 'prompt': - which is what actual JSON/dict serialization looks like, and is what
# datasets/corpus_pipeline.py now rejects at the source via contains_serialized_metadata().
# Re-validated here on both assembled splits from CELL 06 as a second, independent safety net
# before tokenizer training.
from datasets.corpus_pipeline import contains_serialized_metadata, contains_replacement_character

for _split_name, _text in (('train', TRAIN_TEXT_CANDIDATE), ('val', VAL_TEXT_CANDIDATE)):
    print(f'--- Metadata contamination check ({_split_name}) ---')
    print('Contains serialized metadata (quoted "target":/"prompt":/etc key)?',
          contains_serialized_metadata(_text))
    print('Contains U+FFFD replacement character?',
          contains_replacement_character(_text))
    assert not contains_serialized_metadata(_text), f'Real metadata contamination detected in {_split_name} text - fix the source before training.'
    assert not contains_replacement_character(_text), f'{_split_name} text contains U+FFFD - fix the source before training.'

# The bare substring is expected to appear (binary_search's `target` parameter) and is fine -
# only a QUOTED key would be a real problem.
print("\nBare 'target:' substring present in train text (expected, from binary_search - not contamination):",
      'target:' in TRAIN_TEXT_CANDIDATE)
print('\nOK: no real metadata contamination, no replacement-character corruption in train or val text.')

--- Metadata contamination check (train) ---
Contains serialized metadata (quoted "target":/"prompt":/etc key)? False
Contains U+FFFD replacement character? False
--- Metadata contamination check (val) ---
Contains serialized metadata (quoted "target":/"prompt":/etc key)? False
Contains U+FFFD replacement character? False

Bare 'target:' substring present in train text (expected, from binary_search - not contamination): True

OK: no real metadata contamination, no replacement-character corruption in train or val text.


In [ ]:
# CELL 08 - Train (or load) the tokenizer.
#
# should_train_new_tokenizer (training/fresh_mode.py) always returns True while
# fresh_cfg.fresh_tokenizer is True, so a stale vocabulary trained on a different (e.g. the old
# 733-char smoke) corpus can never be silently reused. BPE_VOCAB_PATH was already resolved in
# CELL 06 to the fresh path whenever fresh_tokenizer is set. Exactly one of the two banners
# below always prints, so there is no ambiguity about which happened.
import os
from tokenizer.char_tokenizer import CharTokenizer
from tokenizer.bpe_tokenizer import ByteLevelBPETokenizer
from training.fresh_mode import should_train_new_tokenizer

USE_BPE = True  # set False to use the Phase 1 char tokenizer instead
tokenizer = ByteLevelBPETokenizer(vocab_size=8000) if USE_BPE else CharTokenizer()

if should_train_new_tokenizer(fresh_cfg, tokenizer_path_exists=os.path.exists(BPE_VOCAB_PATH)):
    print('TRAINING NEW TOKENIZER FROM CLEAN DATASET')
    tokenizer.train(TRAIN_TEXT_CANDIDATE)
    tokenizer.save(BPE_VOCAB_PATH)
    print(f'Saved new tokenizer to {BPE_VOCAB_PATH}')
else:
    print('LOADING EXISTING CLEAN TOKENIZER')
    tokenizer.load(BPE_VOCAB_PATH)
    print(f'Loaded existing tokenizer from {BPE_VOCAB_PATH}')

print('Tokenizer type:', tokenizer.tokenizer_type, '| vocab size:', tokenizer.vocab_size)

In [ ]:
# CELL 09 - Tokenizer round-trip test.
# Must preserve Python whitespace/newlines exactly - this is what the model actually trains on.
python_sample = (
    "def quicksort(arr):\n"
    "    if len(arr) <= 1:\n"
    "        return arr\n"
    "\n"
    "    pivot = arr[len(arr) // 2]\n"
)
encoded = tokenizer.encode(python_sample)
decoded = tokenizer.decode(encoded)

print(f'Original:\n{python_sample!r}')
print(f'Decoded:\n{decoded!r}')
print(f'Match: {python_sample == decoded}')
assert python_sample == decoded, 'Tokenizer round-trip failed - do not proceed to training.'

In [ ]:
# CELL 10 - Select a Kodra model configuration (kodra-tiny or kodra-small)
from configs.model_sizes import get_model_size, validate_model_config, estimate_resources
from model.gpt_model import KodraGPT

MODEL_SIZE = 'kodra-tiny'  # one of: kodra-tiny, kodra-small (kodra-base/kodra-medium are roadmap-only)
spec = get_model_size(MODEL_SIZE)
model_cfg = spec.config
model_cfg.vocab_size = tokenizer.vocab_size  # always the (possibly fresh) tokenizer's actual vocab size
validate_model_config(model_cfg)

device = torch.device('cuda' if CUDA_AVAILABLE else 'cpu')
model = KodraGPT(model_cfg).to(device)  # random initialization - no checkpoint is loaded here
print(f'Selected {spec.display_name} on {device}')
print(f'Model vocab_size set from tokenizer: {model_cfg.vocab_size} (tokenizer.vocab_size={tokenizer.vocab_size})')
assert model_cfg.vocab_size == tokenizer.vocab_size, 'Model vocab_size must match the tokenizer vocab_size.'
print('Fresh, randomly-initialized KodraGPT instantiated.')

In [ ]:
# CELL 11 - Exact parameter count (always from the instantiated model, never the roadmap estimate)
from configs.model_sizes import estimate_resources

param_count = model.count_parameters()
resource_estimate = estimate_resources(model_cfg)
print(f'{spec.display_name}: {param_count:,} exact parameters ({param_count/1e6:.2f}M) on {device}')
print(f'Previously trained in this repo: {spec.trained}')
print(f'Rough planning estimate: {resource_estimate["parameters"]:,} params | '
      f'training_vram~{resource_estimate["training_vram_gb"]:.2f}GB | '
      f'inference_vram~{resource_estimate["inference_vram_gb"]:.2f}GB')

In [ ]:
# CELL 12 - Build train/validation dataloaders and the trainer
import sys
import os

# Diagnostics: List everything in /content to see where we are
root_contents = os.listdir('/content')
print("Contents of /content:", root_contents)

# Dynamically search for a folder containing 'configs' or containing 'kodra-core'
found_path = None
search_paths = ['/content/Kodra-ai/kodra-core', '/content/Kodra-ai', '/content']
for p in search_paths:
    if os.path.exists(os.path.join(p, 'configs')):
        found_path = os.path.abspath(p)
        break

if not found_path:
    # Fallback search
    for root, dirs, files in os.walk('/content'):
        if 'configs' in dirs:
            found_path = root
            break

if found_path:
    print(f"Dynamically found repository root containing 'configs' at: {found_path}")
    repo_root = found_path
    os.chdir(repo_root)
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
else:
    raise FileNotFoundError("Could not find the 'configs' directory in /content or its subdirectories. Please re-run CELL 01 to clone the repository.")

from configs.config import TrainingConfig
from datasets.dataset import create_dataloader
from training.trainer import Trainer
from training.utils import set_seed

set_seed(42)

# TRAIN_TEXT / VAL_TEXT come directly from CELL 06's manifest-based, deterministic per-file
# split (not a naive character slice of one blob), matching the split recorded in the manifest.
TRAIN_TEXT = TRAIN_TEXT_CANDIDATE
VAL_TEXT = VAL_TEXT_CANDIDATE

train_cfg = TrainingConfig(batch_size=16, learning_rate=3e-4, max_epochs=10)
train_loader = create_dataloader(TRAIN_TEXT, tokenizer, model_cfg.context_length, train_cfg.batch_size)
val_loader = create_dataloader(VAL_TEXT, tokenizer, model_cfg.context_length, train_cfg.batch_size, shuffle=False)

dataset_manifest_id = f'{manifest.source}-seed{manifest.seed}-{manifest.created_at}'

trainer = Trainer(
    model, train_cfg, train_loader, val_loader=val_loader, device=device,
    tokenizer_type=tokenizer.tokenizer_type, dataset_manifest_id=dataset_manifest_id,
)
print(f'train batches: {len(train_loader)} | val batches: {len(val_loader)}')

Contents of /content: ['.config', 'sample_data']


FileNotFoundError: Could not find the 'configs' directory in /content or its subdirectories. Please re-run CELL 01 to clone the repository.

In [ ]:
import os
print('Files and folders in /content:', os.listdir('/content'))

Files and folders in /content: ['.config', 'sample_data']


In [ ]:
!ls -F

sample_data/


In [ ]:
# CELL 13 - Verify causal next-token shift.
# CodeDataset.__getitem__ must produce input_ids = tokens[:-1], target_ids = tokens[1:] (see
# datasets/dataset.py). Indexes train_loader.dataset directly (bypassing DataLoader shuffling,
# which would otherwise hand back a randomly-ordered chunk, not the deterministic first one) so
# this compares against the exact same chunk the raw token slicing below predicts.
sample_x, sample_y = train_loader.dataset[0]

sample_tokens = tokenizer.encode(TRAIN_TEXT)[: model_cfg.context_length + 1]
expected_x = sample_tokens[:-1]
expected_y = sample_tokens[1:]

print('dataset[0] input[:10]: ', sample_x[:10].tolist())
print('dataset[0] target[:10]:', sample_y[:10].tolist())
print('Expected input[:10] from raw tokens: ', expected_x[:10])
print('Expected target[:10] from raw tokens:', expected_y[:10])

assert sample_x.tolist() == expected_x, 'Causal shift broken: input_ids != tokens[:-1]'
assert sample_y.tolist() == expected_y, 'Causal shift broken: target_ids != tokens[1:]'
assert sample_x.tolist()[1:] == sample_y.tolist()[:-1], 'target is not input shifted by exactly one token'
print('\nOK: causal next-token shift verified (input_ids = tokens[:-1], target_ids = tokens[1:]).')

In [ ]:
# CELL 14 - 20-step GPU smoke training: validates the pipeline end-to-end (forward/backward/step,
# AMP, checkpointing) before committing to a full run. This is NOT the full training loop.
SMOKE_TRAIN_STEPS = 20

smoke_epoch = 0
while trainer.step_count < SMOKE_TRAIN_STEPS:
    smoke_epoch += 1
    smoke_avg_loss = trainer.train_epoch(smoke_epoch, total_steps=SMOKE_TRAIN_STEPS)
    print(f'[smoke] epoch {smoke_epoch} | avg_loss={smoke_avg_loss:.4f} | step={trainer.step_count}')

print(f'Smoke training complete at step {trainer.step_count} (target was {SMOKE_TRAIN_STEPS}).')

In [ ]:
# CELL 15 - Train/validation loss after the smoke run
smoke_val_loss = trainer.evaluate()
print(f'Train loss (last step): {trainer.history[-1]["loss"]:.4f}')
print(f'Validation loss: {smoke_val_loss}')

NameError: name 'trainer' is not defined

In [ ]:
# CELL 16 - Save checkpoint
trainer.save_latest_and_best(CHECKPOINT_DIR, val_loss=smoke_val_loss)
print(f'Saved checkpoint to {os.path.join(CHECKPOINT_DIR, "kodra_gpt_latest.pt")}')

In [ ]:
# CELL 17 - Reload checkpoint into a fresh trainer/model instance.
# assert_checkpoint_reload_is_safe (training/fresh_mode.py) raises if this run's config would
# ever touch the legacy checkpoint lineage - the same guard is unit-tested directly in
# tests/test_fresh_mode.py. The step_count assertion below then proves the reloaded checkpoint
# is the one CELL 16 just saved during THIS run, not an older file.
from training.fresh_mode import assert_checkpoint_reload_is_safe

assert_checkpoint_reload_is_safe(fresh_cfg)

reload_model = KodraGPT(model_cfg).to(device)
reload_trainer = Trainer(
    reload_model, train_cfg, train_loader, val_loader=val_loader, device=device,
    tokenizer_type=tokenizer.tokenizer_type, dataset_manifest_id=dataset_manifest_id,
)

latest_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'kodra_gpt_latest.pt')
reload_trainer.load_checkpoint(latest_checkpoint_path)
print(f'Reloaded trainer from checkpoint: {latest_checkpoint_path}')
print(f'Reloaded step_count: {reload_trainer.step_count} (expected {trainer.step_count})')
assert reload_trainer.step_count == trainer.step_count, 'Checkpoint reload lost step_count.'
print('Confirmed: reloaded checkpoint matches the one saved earlier in THIS fresh run.')

In [ ]:
# CELL 18 - Resume training for at least 5 more steps on the reloaded trainer.
# Training runs in whole epochs (see Trainer.train_epoch), so once the loader has more than
# one batch, resuming can overshoot an exact step target by up to (len(train_loader) - 1)
# steps - this loop guarantees AT LEAST RESUME_STEPS additional steps happened, which is the
# real, achievable contract of an epoch-granularity trainer.
RESUME_STEPS = 5
resume_target = reload_trainer.step_count + RESUME_STEPS
steps_before_resume = reload_trainer.step_count
resume_epoch = 0
while reload_trainer.step_count < resume_target:
    resume_epoch += 1
    resume_avg_loss = reload_trainer.train_epoch(resume_epoch, total_steps=resume_target)
    print(f'[resume] epoch {resume_epoch} | avg_loss={resume_avg_loss:.4f} | step={reload_trainer.step_count}')

steps_resumed = reload_trainer.step_count - steps_before_resume
print(f'Resumed training to step {reload_trainer.step_count} ({steps_resumed} steps, target was >= {RESUME_STEPS}).')
assert steps_resumed >= RESUME_STEPS, 'Resume did not advance by at least RESUME_STEPS.'

In [ ]:
# CELL 19 - Evaluate generation/syntax.
# CodeGenerator.generate() returns prompt + continuation (documented contract, see
# inference/generator.py); python_parses() is called on that exact string with no
# post-processing, so the syntax score is never artificially inflated. A low/0% parse rate here
# on a smoke-trained model is expected (see the notebook intro) - it is a true reflection of an
# undertrained model, not an evaluator bug.
from evaluation.evaluator import full_evaluation_report
from inference.generator import CodeGenerator
import json as _json

report = full_evaluation_report(reload_model, tokenizer, device, val_loader)
print(_json.dumps(report, indent=2, default=str))

generator = CodeGenerator(reload_model, tokenizer, device)
sample = generator.generate('def quicksort(arr):', max_new_tokens=64, temperature=0.7, top_k=40)
print('\nSample completion (raw generator output, prompt + continuation):')
print(repr(sample))

In [ ]:
# CELL 20 - (Optional) Back up checkpoints to Google Drive. Only runs if Drive is mounted (see
# the optional cell above) and BACKUP_TO_DRIVE is set True.
BACKUP_TO_DRIVE = False
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/kodra_checkpoints_backup'

if BACKUP_TO_DRIVE:
    import shutil
    if not os.path.isdir('/content/drive'):
        print('Google Drive is not mounted - skipping backup. Mount it in CELL 05 first.')
    else:
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        for fname in os.listdir(CHECKPOINT_DIR):
            shutil.copy2(os.path.join(CHECKPOINT_DIR, fname), os.path.join(DRIVE_BACKUP_DIR, fname))
        print(f'Backed up checkpoints from {CHECKPOINT_DIR} to {DRIVE_BACKUP_DIR}')
else:
    print('BACKUP_TO_DRIVE is False - skipping Drive backup.')

In [ ]:
# CELL 21 - FULL TRAINING - RUN MANUALLY ONLY
#
# Everything above only validated the pipeline (20-step smoke test + 5-step resume) on
# APPROVED_DATASET_DIR. Before any serious (non-smoke) training run, this cell runs the
# mandatory readiness gate (training/readiness.py) and refuses to proceed if the assembled
# corpus/tokenizer is not ready, independently of RUN_FULL_TRAINING. Per the root-cause
# diagnosis in this notebook's intro, the checkpoint produced by CELLS 14-18 is a
# diagnostic/legacy checkpoint only and is never used as a production starting point -
# FRESH_TRAINING guarantees a clean model/tokenizer/checkpoint lineage (see CELL 06/08/17).
from training.readiness import check_serious_training_gate

RUN_FULL_TRAINING = False  # defaults False so re-running the whole notebook ("Run All") never
                            # triggers a full run; flip to True yourself only after the gate below passes

readiness_report = check_serious_training_gate(
    TRAIN_TEXT_CANDIDATE, VAL_TEXT_CANDIDATE, manifest.total_token_estimate, tokenizer=tokenizer,
)
print(readiness_report.summary())

if not RUN_FULL_TRAINING:
    raise RuntimeError(
        'Full training is gated. Set RUN_FULL_TRAINING = True above and re-run this '
        'cell to start the full training loop from scratch on a validated corpus.'
    )

if not readiness_report.passed:
    raise RuntimeError(
        'Serious-training readiness gate failed - refusing to start full training:\n'
        + readiness_report.summary()
    )

total_steps = train_cfg.max_epochs * len(train_loader)
for epoch in range(1, train_cfg.max_epochs + 1):
    avg_loss = trainer.train_epoch(epoch, total_steps=total_steps)
    val_loss = trainer.evaluate()
    trainer.save_latest_and_best(CHECKPOINT_DIR, val_loss=val_loss)
    tps = trainer.history[-1]['tokens_per_sec'] if trainer.history else 0.0
    print(f'Epoch {epoch}/{train_cfg.max_epochs} | avg_loss={avg_loss:.4f} | '
          f'val_loss={val_loss} | step={trainer.step_count} | tokens/sec={tps:.0f}')

ModuleNotFoundError: No module named 'training'